In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType, IntegerType, TimestampType

# Hàm làm sạch dữ liệu số thập phân (Decimal)
def clean_decimal(c, p=38, s=8):
    cleaned = regexp_replace(col(c), "[^0-9.]", "")
    return nullif(cleaned, lit("")).alias(c)

# Hàm làm sạch dữ liệu số nguyên (Integer)
def clean_int(c):
    cleaned = regexp_replace(col(c), "[^0-9]", "")
    return when(
        length(cleaned) == 0,
        None
    ).otherwise(
        cleaned.cast(IntegerType())
    ).alias(c)
# Hàm xử lý Timestamp đặc thù (dd/MM/yyyy HH:mm)
def clean_timestamp(c):
    return coalesce(
        to_timestamp(c, "dd/MM/yyyy HH:mm"),
        to_timestamp(c, "yyyy-MM-dd HH:mm:ss"),
        to_timestamp(c) 
    )

# 1. Đọc dữ liệu từ nguồn (CSV)
df = (
    spark.read
    .option("header", "true")
    .option("encoding", "UTF-8")
    .option("inferSchema", "false")
    .csv("/Volumes/mb_poc/data_raw/uc2/QUEUING.csv")
)

# 2. Transform dữ liệu theo Schema của RPT_TRANSACTION_QUEUING
# Cập nhật phần select để trim tên cột và xử lý cột cuối
df_tgt = (
    df.select(
        clean_timestamp("TXN_DT").alias("TXN_DT"),
        col("CNC2_CODE").alias("CNC2_CODE"),
        col("CNC2_NAME").alias("CNC2_NAME"),
        col("CNC1_CODE").alias("CNC1_CODE"),
        col("CNC1_NAME").alias("CNC1_NAME"),
        col("TICKET_NBR").alias("TICKET_NBR"),
        col("CST_CODE").alias("CST_CODE"),
        col("CST_NM").alias("CST_NM"),
        col("TXN_TP").alias("TXN_TP"),
        col("CST_SEG").alias("CST_SEG"),
        col("TXN_NM").alias("TXN_NM"),
        col("TTL_NM").alias("TTL_NM"),
        col("SRV_USR_ID").alias("SRV_USR_ID"),
        col("SRV_USR_NM").alias("SRV_USR_NM"),
        clean_int("SRV_STALL").alias("SRV_STALL"),
        col("TICKET_TIME_SLOT").alias("TICKET_TIME_SLOT"),
        clean_timestamp("TICKET_TIME").alias("TICKET_TIME"),
        clean_timestamp("START_SRV_TIME").alias("START_SRV_TIME"),
        clean_timestamp("END_TXN_TIME").alias("END_TXN_TIME"),
        clean_decimal("TOTAL_WAIT_TIME", 10, 2).alias("TOTAL_WAIT_TIME"),
        clean_decimal("SRV_DURATION", 10, 2).alias("SRV_DURATION"),
        col("END_STATUS").alias("END_STATUS"),
        col("REMARK").alias("REMARK"),
        col("BOOKING_TICKET").alias("BOOKING_TICKET"),
        col("TRANSFER_TICKET").alias("TRANSFER_TICKET"),
        col("TICKET_FLOW").alias("TICKET_FLOW"),
        col("MY_MB_APP_FLOW").alias("MY_MB_APP_FLOW")
    )
)
#df_tgt.display()
# 3. Ghi dữ liệu vào bảng Gold
df_tgt.write.mode("append").insertInto("mb_poc.gold.rpt_transaction_queuing")